[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/04_machine_learning/16_feature_engineering.ipynb)

# 📓 Notebook 16 — Feature Engineering

> **Module:** Machine Learning · **Estimated time:** 55–70 min · **Difficulty:** Intermediate

> *"Most of what looks like a model problem is actually a feature problem."* — every senior ML engineer, eventually.

A model can only learn what the features tell it. Better features almost always beat a fancier model — and they are a *much* cheaper investment. This notebook walks through the feature-engineering toolkit on a small tabular dataset, with a special focus on the single most damaging mistake: **target leakage**.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Choose the right **encoding** for categorical features (one-hot, ordinal, target encoding — and when each is wrong).
2. Apply **scaling** correctly (when it matters, when it doesn't, where the leak hides).
3. Extract useful information from **datetime** columns.
4. Build **interaction** and **ratio** features that lift linear models.
5. Spot and prevent **target leakage** before it ships.
6. Use **`SelectKBest`** and feature importance to prune features.
7. Assemble a complete leak-free **pipeline** that's ready for cross-validation.

## ✅ Prerequisites

NB 14 (sklearn basics) and NB 15 (model evaluation — especially the Pipeline discipline).

## 1. Setup — a slightly richer dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
RANDOM_STATE = 42

**Build a richer churn dataset (with signup_date and region)**

In [ ]:
# A richer churn dataset that includes a signup_date so we can show date features
rng = np.random.default_rng(RANDOM_STATE)
n = 600
mrr_eur         = rng.lognormal(mean=6.5, sigma=0.7, size=n).clip(50, 50_000)
months_active   = rng.integers(1, 60, size=n)
support_tickets = rng.poisson(lam=2.5, size=n)
plan            = rng.choice(["Free","Standard","Premium","Enterprise"],
                              size=n, p=[0.15,0.45,0.30,0.10])
last_login_days = rng.integers(0, 90, size=n)
signup_date     = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 365, size=n), unit="D")
region          = rng.choice(["EU", "US", "APAC", "LATAM"], size=n, p=[0.4, 0.4, 0.15, 0.05])

plan_adj = np.array([{"Free":-0.45,"Standard":0.0,"Premium":0.15,"Enterprise":0.35}[p] for p in plan])
happiness = (-0.40*(support_tickets-2.5)/2.0 -0.80*(last_login_days-45)/30.0
             +0.35*(np.log1p(mrr_eur)-6.5)/1.5 +0.30*(months_active-30)/20.0
             + plan_adj + rng.normal(0,0.30,n))
churn_prob = 1/(1+np.exp(-(-1.6 - 2.5*happiness)))
churned = (rng.random(n) < churn_prob).astype(int)

df = pd.DataFrame({"mrr_eur":mrr_eur, "months_active":months_active,
                    "support_tickets":support_tickets, "plan":plan,
                    "last_login_days":last_login_days, "signup_date":signup_date,
                    "region":region, "churned":churned})
print(df.head())
print(f"\nShape: {df.shape}, churn rate: {df['churned'].mean():.1%}")

## 2. Categorical encoding — three options, three trade-offs

| Encoding | When to use | When NOT to use |
|---|---|---|
| **One-hot** | Few categories, no ordering | Hundreds of categories — explodes dimensionality |
| **Ordinal** | True ordering (`small/medium/large`) | Nominal categories — model will infer false ordering |
| **Target / mean encoding** | Many categories, you have enough data | Small data — leaks the target value into features |

Let's see each.

In [ ]:
# (a) One-hot — safe default for nominal categorical with few values
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_oh = oh.fit_transform(df[["plan"]])
print("One-hot encoding of 'plan':")
print(pd.DataFrame(X_oh, columns=oh.get_feature_names_out(["plan"])).head())


### 🔬 What actually happens in one-hot encoding?

The line `oh.fit_transform(df[["plan"]])` above quietly did two jobs. Let's slow it right down, because once you *see* the mechanics, every encoder in sklearn stops being magic.

**The problem it solves.** A model is just arithmetic — it multiplies, adds, and compares **numbers**. It cannot multiply by the string `"Premium"`. So *every* categorical column has to become numbers somehow. One-hot is the safe default way to do that for **nominal** categories (categories with no natural order).

**The mechanic.** Take one column with `k` distinct categories. One-hot replaces it with `k` brand-new columns — one per category — each holding only `0` or `1`. A row gets a `1` in the single column matching its category, and `0` everywhere else. Exactly one `1` per row — the column is "one-hot."

```text
   plan                       plan_Free  plan_Standard  plan_Premium  plan_Enterprise
 ┌──────────┐                ┌─────────┬─────────────┬────────────┬───────────────┐
 │ Standard │   one-hot      │    0    │      1      │     0      │       0       │
 │ Premium  │  ──────────▶   │    0    │      0      │     1      │       0       │
 │ Free     │                │    1    │      0      │     0      │       0       │
 │ Premium  │                │    0    │      0      │     1      │       0       │
 └──────────┘                └─────────┴─────────────┴────────────┴───────────────┘
   1 text column                  k = 4 numeric columns, exactly one 1 per row
```

The proof cell below builds this by hand with plain pandas, then confirms sklearn produces the identical thing.


In [ ]:
# 🧪 PROOF: one-hot is just "make one 0/1 column per category"
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

tiny_oh = pd.DataFrame({"plan": ["Standard", "Premium", "Free", "Premium"]})
print("Original column:")
print(tiny_oh, "\n")

# --- By hand: for each category, a column that is 1 where plan == that category ---
by_hand = pd.DataFrame({
    "plan_Free":       (tiny_oh["plan"] == "Free").astype(int),
    "plan_Standard":   (tiny_oh["plan"] == "Standard").astype(int),
    "plan_Premium":    (tiny_oh["plan"] == "Premium").astype(int),
})
print("Built by hand (one 0/1 column per category):")
print(by_hand, "\n")

# Every row has exactly ONE 1 — that's what 'one-hot' means:
print("Sum across each row (always 1):", by_hand.sum(axis=1).tolist())

# --- pd.get_dummies does the same in one call ---
print("\npd.get_dummies (note the int columns match by_hand):")
print(pd.get_dummies(tiny_oh["plan"], prefix="plan").astype(int))

# --- sklearn's OneHotEncoder: the same numbers, but it LEARNS the categories in fit() ---
oh_demo = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
arr_demo = oh_demo.fit_transform(tiny_oh[["plan"]])
print("\nsklearn OneHotEncoder learned these categories during fit():")
print(" ", list(oh_demo.categories_[0]))
print("and produced the same 0/1 matrix:")
print(pd.DataFrame(arr_demo.astype(int), columns=oh_demo.get_feature_names_out(["plan"])))


### Why not just number the categories `0, 1, 2, 3`?

It's tempting to skip one-hot and map `Free→0, Standard→1, Premium→2, Enterprise→3` in a single column. For **nominal** data this quietly breaks the model, and it's worth knowing exactly why.

A model reads those integers as a **number line**. It now believes:

```text
   Free      Standard     Premium      Enterprise
    0────────────1────────────2────────────3
   "Enterprise is 3× Free"   "Premium sits between Standard and Enterprise"
   "the gap Free→Standard equals the gap Premium→Enterprise"
```

None of that is true — these are just *names*. You've invented a fake order and fake distances the model will dutifully try to learn from.

| | **Ordinal-as-integer** (`0,1,2,3` in one column) | **One-hot** (`k` separate 0/1 columns) |
|---|---|---|
| Implies an order? | **Yes** — `0 < 1 < 2 < 3` | **No** — every category is its own axis |
| Implies distances? | **Yes** — gaps are comparable | **No** — all equidistant |
| Safe for **nominal** (`region`, `plan`)? | ❌ injects fake structure | ✅ correct default |
| Safe for **truly ordered** (`small<medium<large`)? | ✅ that's `OrdinalEncoder`'s job | ✅ but throws away the order |
| Column count | 1 | `k` (or `k-1`) |

> 🧠 **Mental model.** One-hot gives each category its **own private switch** — flipping "Premium" on says *nothing* about "Free." Integer-encoding forces every category onto **one shared dial**, and a dial always implies order and distance. Use integers only when the order is real.

### Why you'll sometimes see `k − 1` columns ("drop first")

With `k` one-hot columns, any one column is **redundant**: if `plan_Free`, `plan_Standard`, `plan_Premium` are all `0`, the row *must* be `Enterprise` — the last column carries no new information. Dropping one (`drop="first"`) avoids perfect **collinearity**, which matters for linear models. Tree models and regularised models don't care, so the safe default is to keep all `k`. The cell below shows both.


In [ ]:
# 🧪 PROOF: k columns vs k-1 columns (drop="first") carry the SAME information
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

tiny_drop = pd.DataFrame({"plan": ["Free", "Standard", "Premium", "Enterprise"]})

full = OneHotEncoder(sparse_output=False).fit_transform(tiny_drop[["plan"]])
drop = OneHotEncoder(sparse_output=False, drop="first").fit_transform(tiny_drop[["plan"]])

print("All k = 4 columns:")
print(pd.DataFrame(full.astype(int),
                   columns=OneHotEncoder(sparse_output=False)
                           .fit(tiny_drop[["plan"]]).get_feature_names_out(["plan"])), "\n")

print("Only k-1 = 3 columns (drop='first'):")
print(pd.DataFrame(drop.astype(int)), "\n")

# The dropped category is recoverable: it's the row that is all-zeros.
print("All-zero rows in the k-1 version are the dropped category:")
print((drop.sum(axis=1) == 0))   # the 'Enterprise' row (alphabetically first dropped) -> all zeros


In [ ]:
# (b) Ordinal — when there IS an order
plan_order = [["Free", "Standard", "Premium", "Enterprise"]]
ord_enc = OrdinalEncoder(categories=plan_order)
df["plan_ordinal"] = ord_enc.fit_transform(df[["plan"]]).astype(int)
print(df[["plan", "plan_ordinal"]].drop_duplicates().sort_values("plan_ordinal"))


> ⚠️ **The classic ordinal mistake.** A column with `["red", "green", "blue"]` is *not* ordinal — `green` is not "between" red and blue. Using ordinal encoding on nominal data injects a fictitious ordering the model will dutifully try to learn.

In [ ]:
# (c) Target encoding — DANGEROUS without care
# Goal: replace each plan with the mean churn rate for that plan (on training data only!)

def target_encode(train_df, valid_df, col, target):
    """Compute target encoding on training data only; apply to validation."""
    means = train_df.groupby(col)[target].mean()
    return train_df[col].map(means), valid_df[col].map(means)


train_df, valid_df = train_test_split(df, test_size=0.25, random_state=RANDOM_STATE,
                                      stratify=df["churned"])
te_train, te_valid = target_encode(train_df, valid_df, "plan", "churned")
print("Target encoding learned on training data:")
print(train_df.groupby("plan")["churned"].mean().round(3))
print("\nFirst 5 encoded training values:")
print(te_train.head())


> 🎯 **Why target encoding leaks if you're sloppy.** If you compute the mean on *all* the data (including future test rows), the test rows see their own target through the encoding. The CV score then looks fantastic — and production accuracy collapses.
>
> **Fix:** compute target encoding inside a CV fold, on training rows only. The `sklearn-contrib` package `category_encoders` does this safely via `TargetEncoder(cv=5)`.

## 3. Scaling — when it matters, when it doesn't

| Model family | Needs scaling? | Why |
|---|---|---|
| Logistic Regression, SVM, k-NN, PCA, neural nets | **Yes** | Distance / gradient-based: unscaled features dominate |
| Decision Tree, Random Forest, Gradient Boosting | **No** | Tree splits are invariant to monotone transformations |
| Naive Bayes | Depends | Discretisation-friendly variants don't need it |

For our churn data, `mrr_eur` ranges 50–50,000 while `support_tickets` is 0–10. A logistic regression sees `mrr_eur` as ~1000× more important *just because of scale*.

In [ ]:
# Demonstrate why scaling matters for LR but not for RF
NUM = ["mrr_eur", "months_active", "support_tickets", "last_login_days"]
X = df[NUM]
y = df["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           random_state=RANDOM_STATE, stratify=y)

# LR unscaled
lr_raw = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
# LR scaled
lr_scaled = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
# RF (no scaling needed)
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X_tr, y_tr)

print(f"LR no scaling : test score = {lr_raw.score(X_te, y_te):.3f}")
print(f"LR with scaler: test score = {lr_scaled.score(X_te, y_te):.3f}")
print(f"RF (no scale) : test score = {rf.score(X_te, y_te):.3f}")


**See what the scaler does.** The scores above say scaling helps logistic regression but not the forest. The *why* is visible in one picture: on the raw scale `mrr_eur` (thousands) utterly dwarfs `support_tickets` (single digits), so a distance/gradient model effectively only "sees" MRR. `StandardScaler` re-expresses every feature as *standard deviations from its own mean*, putting them on a common axis — note the y-scale changes from **log** (left) to a tidy ±3 (right).


In [ ]:
raw = X_tr.copy()
scaled = pd.DataFrame(StandardScaler().fit_transform(X_tr), columns=NUM)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
sns.boxplot(data=raw, ax=axes[0], color="#4C72B0", fliersize=2)
axes[0].set_yscale("log")
axes[0].set(title="Raw features — wildly different scales (log y)", ylabel="value (log)")
sns.boxplot(data=scaled, ax=axes[1], color="#55A868", fliersize=2)
axes[1].axhline(0, color="grey", lw=1)
axes[1].set(title="After StandardScaler — comparable spread", ylabel="standard deviations")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Why distance/gradient models need scaling — but trees don't", y=1.03)
plt.tight_layout(); plt.show()


**Three scaling rules:**

1. **Use `StandardScaler`** by default — zero mean, unit variance.
2. **Use `MinMaxScaler`** when you have a known bounded range or feed images / pixels.
3. **Put the scaler inside the `Pipeline`** — never `fit_transform(X)` on the whole dataset before splitting. (See NB 15 for the leak pattern.)

### 🔬 What a scaler actually *learns* — and why the test set must never be in the room

Rule 3 above said *"never `fit_transform(X)` on the whole dataset before splitting."* That sounds like fussy hygiene. It isn't — it's the difference between an honest score and a fantasy. To see why, you have to know that **a `StandardScaler` has memory.**

`StandardScaler` turns each feature into `(x − mean) / std`. But *which* mean and *which* std? It doesn't know them in advance — it **learns** them from whatever data you hand to `.fit()`, and stashes them on the scaler as `scaler.mean_` and `scaler.scale_` (the learned std). The trailing `_` is sklearn's universal marker for *"this was learned during fit"*. After that, every `.transform()` reuses those exact stored numbers.

So the call splits cleanly into two jobs:

```text
        ┌─────────────── .fit(X_train) ───────────────┐
        │  LEARN  mean_ and std  FROM X_train only     │   <-- looks at data, memorises stats
        └──────────────────────────────────────────────┘
                              │  stored on the scaler
              ┌───────────────┴────────────────┐
              ▼                                 ▼
   .transform(X_train)                .transform(X_test)
   (x - mean_)/std                    (x - mean_)/std       <-- SAME numbers, no peeking
```

**The leak.** If you instead call `scaler.fit(X_all)` (or fit a fresh scaler on the test set), the learned mean/std are computed *using the test rows*. Information about the test set has now flowed backwards into your preprocessing. Your model is graded on data it has already, indirectly, seen.

> 🧠 **Mental model.** The test set stands in for **genuinely unseen, future data** — rows that don't exist yet when you train. You can't compute a mean over rows that don't exist yet. So the scaler is only ever allowed to learn from the past (train), then apply that frozen rule to the future (test). **If your scaler peeked at the test set, the test set isn't unseen anymore — and your score is a lie.**

The cell below proves it on tiny inline data: same split, two ways, and the leaked stats come out *measurably different*.


In [ ]:
# 🧪 PROOF: fitting on TRAIN-only vs ALL data gives different scaler stats (= the leak)
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

rng_demo = np.random.default_rng(0)
# One feature. Train values are small; the (future) test values run large on purpose,
# so "peeking" at them visibly shifts the learned mean/std.
X_demo = np.concatenate([rng_demo.normal(10, 1, size=8),     # these will become TRAIN
                    rng_demo.normal(40, 5, size=4)]     # these will become TEST
                   ).reshape(-1, 1)
X_train, X_test = X_demo[:8], X_demo[8:]      # deterministic split for a clean comparison

# ----- ✅ RIGHT WAY: fit on TRAIN only, then transform BOTH with those stats -----
# (sklearn stores the learned std in `.scale_`; `.mean_` holds the learned mean)
right = StandardScaler().fit(X_train)
print("RIGHT  (fit on X_train):  mean_ = %.3f   std (scale_) = %.3f" % (right.mean_[0], right.scale_[0]))
Xtr_r = right.transform(X_train)
Xte_r = right.transform(X_test)     # test uses the TRAIN stats — as it must
print("   train -> mean ~0, std ~1 :  mean=%.3f std=%.3f" % (Xtr_r.mean(), Xtr_r.std()))
print("   test  -> NOT 0/1 (correct! it is unseen): mean=%.3f std=%.3f" % (Xte_r.mean(), Xte_r.std()))

# ----- ❌ WRONG WAY: fit on ALL data (train + test) before splitting -----
leaky = StandardScaler().fit(X_demo)     # the scaler has now SEEN the test rows
print("\nLEAKY  (fit on X_all):    mean_ = %.3f   std (scale_) = %.3f" % (leaky.mean_[0], leaky.scale_[0]))

print("\nDifference caused purely by letting the scaler peek at the test set:")
print("   mean_ shifted by %.3f,  std shifted by %.3f" %
      (abs(leaky.mean_[0] - right.mean_[0]), abs(leaky.scale_[0] - right.scale_[0])))
print("   => the LEAKY transform of X_train was computed using test information it should never see.")


### The leak-proof pattern: put the scaler inside a `Pipeline`

Doing the train-only fit by hand is easy to get wrong — especially under cross-validation, where the train/test boundary moves on **every fold**. The robust fix is to never split the scaler from the model: bundle them in a `Pipeline`.

```text
            Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])

   .fit(X_train, y_train)                     .predict(X_test)
   ────────────────────                       ────────────────
   scaler.fit(X_train)  ── learns mean_/std   scaler.transform(X_test)  ── reuses them
   scaler.transform(X_train)                  model.predict(...)
   model.fit(...)
```

When you call `pipe.fit(X_train, ...)`, the pipeline fits the scaler on **that** training data only; when you `pipe.predict`/`pipe.score` on the test fold, it *transforms* with the already-learned stats. During `cross_val_score`, this happens correctly and automatically for **each fold** — the scaler can never see the held-out rows. The proof cell below shows the pipeline reproduces the hand-done RIGHT WAY exactly.


In [ ]:
# 🧪 PROOF: a Pipeline does the train-only fit for you, identically to the manual RIGHT WAY
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

rng_demo = np.random.default_rng(1)
X_demo = np.concatenate([rng_demo.normal(10, 1, size=8), rng_demo.normal(40, 5, size=4)]).reshape(-1, 1)
y_demo = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1])
X_train, X_test, y_train = X_demo[:8], X_demo[8:], y_demo[:8]

# Manual RIGHT WAY
manual = StandardScaler().fit(X_train)

# Pipeline way — fitting the pipeline fits the scaler on X_train ONLY, behind the scenes
pipe = Pipeline([("scaler", StandardScaler()),
                 ("model", LogisticRegression())]).fit(X_train, y_train)
pipe_scaler = pipe.named_steps["scaler"]

print("Manual   scaler stats:  mean_=%.4f  std=%.4f" % (manual.mean_[0], manual.scale_[0]))
print("Pipeline scaler stats:  mean_=%.4f  std=%.4f" % (pipe_scaler.mean_[0], pipe_scaler.scale_[0]))
print("Identical? ", np.allclose(manual.mean_,  pipe_scaler.mean_) and
                      np.allclose(manual.scale_, pipe_scaler.scale_))
print("\n=> The Pipeline learned mean_/std from X_train alone — the test set never leaked in.")


## 4. Datetime features — turning a date into many signals

A `signup_date` column on its own is useless to a model. But it contains many useful features hiding inside it: day of week, month, year, tenure, season, business-day-or-not.

In [ ]:
# Build a bunch of useful features from a single datetime column
ref = pd.Timestamp("2024-06-01")   # the "as-of" date — usually "now" in production

dfx = df.copy()
dfx["signup_year"]   = dfx["signup_date"].dt.year
dfx["signup_month"]  = dfx["signup_date"].dt.month
dfx["signup_dow"]    = dfx["signup_date"].dt.dayofweek         # 0 = Monday
dfx["signup_q"]      = dfx["signup_date"].dt.quarter
dfx["tenure_days"]   = (ref - dfx["signup_date"]).dt.days
dfx["is_weekend_signup"] = (dfx["signup_dow"] >= 5).astype(int)

# Cyclical encoding for month (so December and January end up close together)
dfx["signup_month_sin"] = np.sin(2*np.pi*dfx["signup_month"]/12)
dfx["signup_month_cos"] = np.cos(2*np.pi*dfx["signup_month"]/12)

print(dfx[["signup_date", "signup_year", "signup_month", "tenure_days",
            "signup_dow", "is_weekend_signup",
            "signup_month_sin", "signup_month_cos"]].head())


**Why cyclical encoding?** A linear model treats month=1 and month=12 as 11 apart — but they're really 1 apart in cyclic time. Sin / cos pairs encode "circular distance" correctly. Same trick works for hours of the day, days of the week, and any periodic phenomenon.

> 💡 **Tenure features are almost always useful.** `tenure_days` (days since signup) is a strong churn predictor on its own — even when you have rich behavioural features.

## 5. Interaction and ratio features — help linear models see non-linearity

In [ ]:
# Two examples:
# (a) Ratio: tickets per month of tenure -- normalises by how long they've been a customer
dfx["tickets_per_month"] = dfx["support_tickets"] / dfx["months_active"].clip(lower=1)

# (b) Interaction: is this a Free-plan customer with low engagement?
dfx["free_and_inactive"] = ((dfx["plan"] == "Free") & (dfx["last_login_days"] > 30)).astype(int)

# (c) Log of a heavy-tailed feature -- spreads the values more linearly
dfx["log_mrr"] = np.log1p(dfx["mrr_eur"])

print(dfx[["mrr_eur", "log_mrr", "support_tickets", "months_active",
            "tickets_per_month", "free_and_inactive"]].head())


> 🎯 **Heavy-tailed features (MRR, latency, follower counts) almost always benefit from `log1p`.** It pulls the long right tail toward the centre so models can use it linearly.

## 6. Target leakage — the bug that ruins ML projects

A feature is **leaky** when it contains information that *wouldn't be available at prediction time*. Three common ways leakage sneaks in:

1. **The label disguised as a feature.** `did_customer_call_to_cancel` predicting `churned` perfectly — because it's basically the label.
2. **Future information.** `total_lifetime_spend` includes spend that happened *after* the churn event.
3. **Group statistics computed on the whole dataset.** `avg_revenue_for_this_plan` computed on train+test together leaks test means.

Let's simulate a subtle leak and watch it produce suspiciously good metrics.

In [ ]:
# Build a "leaky" feature that's slightly informed by the future
# Pretend: total_engagement is a feature we compute, but it secretly used some
# post-event activity. So it correlates with churned more strongly than it should.
np.random.seed(0)
df["total_engagement"] = (
    50 * (1 - df["churned"])            # ← leak: directly uses the target
    + 5 * df["months_active"]
    + np.random.normal(0, 5, len(df))
)

X_leak = df[["mrr_eur", "months_active", "support_tickets", "total_engagement"]]
X_clean = df[["mrr_eur", "months_active", "support_tickets"]]

pipe = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))])
print(f"With leak     : {cross_val_score(pipe, X_leak,  df['churned'], cv=5).mean():.3f}")
print(f"Without leak  : {cross_val_score(pipe, X_clean, df['churned'], cv=5).mean():.3f}")


**How to *see* a leak.** The 10-point jump is the smell; this is the picture. Plot each feature's distribution split by the target. A **legitimate** feature overlaps a lot between churned and stayed (it carries *some* signal). A **leaky** feature separates the two classes almost perfectly — because it secretly contains the answer.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, col, kind in [(axes[0], "months_active", "legitimate feature"),
                      (axes[1], "total_engagement", "LEAKY feature")]:
    for cls, color, lbl in [(0, "#4C72B0", "stayed"), (1, "#C44E52", "churned")]:
        ax.hist(df.loc[df["churned"] == cls, col], bins=25, alpha=0.6, color=color, label=lbl)
    ax.set(title=f"{col}\n({kind})", xlabel=col)
    ax.legend()
axes[0].set_ylabel("number of customers")
fig.suptitle("A leaky feature separates the classes too cleanly to be true", y=1.03)
plt.tight_layout(); plt.show()

print("Correlation with target:")
print(f"  months_active     : {df['months_active'].corr(df['churned']):+.3f}  (modest — believable)")
print(f"  total_engagement  : {df['total_engagement'].corr(df['churned']):+.3f}  (huge — investigate!)")


**That's the smell.** A new feature pushes CV accuracy by 10+ points with no extra data and no model change → almost certainly a leak.

**Checklist before trusting a "great" feature:**

- Could this value have been measured *before* the event you're predicting?
- Is this an aggregate computed on the whole dataset (instead of training only)?
- Does this column rank-correlate with the target above 0.9 by itself?

If any answer is "yes / I'm not sure", investigate before you ship.

## 7. Feature selection — keeping only what helps

In [ ]:
# Univariate F-statistic ranking of numerical features against churn
NUM = ["mrr_eur", "log_mrr", "months_active", "support_tickets",
       "last_login_days", "tickets_per_month"]
sel = SelectKBest(score_func=f_classif, k="all").fit(dfx[NUM], dfx["churned"])

rank = pd.DataFrame({"feature": NUM,
                      "F_score": sel.scores_,
                      "p_value": sel.pvalues_}).sort_values("F_score", ascending=False)
print(rank.round(4))


**The ranking as a picture.** The table sorts the features by univariate F-score; the chart makes the gaps between them legible at a glance — which features carry strong standalone signal, and where the cliff is.


In [ ]:
order = rank.sort_values("F_score")
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(order["feature"], order["F_score"], color="#DD8452")
ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=10)
ax.set(title="Univariate feature ranking — F-score vs churn", xlabel="ANOVA F-score (higher = stronger standalone signal)")
sns.despine(left=True)
plt.tight_layout(); plt.show()


**Two practical takeaways:**

- High F-score + low p-value = feature carries signal *on its own*.
- A low score doesn't mean "drop it" — it might still be useful **in combination** with other features. Tree-based models exploit interactions you can't see from a univariate test.

> 💡 **Random-forest feature importance** (`rf.feature_importances_`) is a complementary view: it tells you which features the model *actually used*, including interactions.

## 8. A complete leak-free pipeline

In [ ]:
# All preprocessing inside a single Pipeline / ColumnTransformer
NUM_FEATURES = ["mrr_eur", "log_mrr", "months_active", "support_tickets",
                 "last_login_days", "tickets_per_month", "tenure_days",
                 "signup_month_sin", "signup_month_cos", "is_weekend_signup"]
CAT_FEATURES = ["plan", "region"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
], remainder="drop")

pipe = Pipeline([("prep", preprocess),
                  ("model", LogisticRegression(max_iter=1000))])

# Cross-validation with stratification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, dfx, dfx["churned"], cv=cv, scoring="roc_auc")
print(f"5-fold ROC AUC: {scores.round(3)}")
print(f"Mean ± std   : {scores.mean():.3f} ± {scores.std():.3f}")


**Read this pipeline one more time.** Every transformer is fit on each CV fold's training portion only, then applied to that fold's validation portion. There is no path through which validation data could influence a transformer's parameters. *This is what "no leakage" means in code.*

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add tenure features

Build a new feature `tenure_months` (`tenure_days / 30`) and a categorical feature `is_long_tenure` (`tenure_months > 12`). Add them to the pipeline and report whether they improve the CV score.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
dfx["tenure_months"]  = dfx["tenure_days"] / 30
dfx["is_long_tenure"] = (dfx["tenure_months"] > 12).astype(int)

NEW_NUM = NUM_FEATURES + ["tenure_months", "is_long_tenure"]
prep2 = ColumnTransformer([
    ("num", StandardScaler(), NEW_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])
pipe2 = Pipeline([("prep", prep2), ("model", LogisticRegression(max_iter=1000))])
scores2 = cross_val_score(pipe2, dfx, dfx["churned"], cv=cv, scoring="roc_auc")
print(f"Original: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"+ tenure: {scores2.mean():.3f} ± {scores2.std():.3f}")
```

If the lift is within the standard deviation, it's noise. The honest rule: a feature should beat the noise by at least 2× the std before you keep it.
</details>

### Exercise 2 — ⭐⭐ Spot the leak

Look at this proposed feature for predicting churn:

```python
df["avg_churn_rate_for_plan"] = df.groupby("plan")["churned"].transform("mean")
```

Is it leaky? If yes, fix it.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

**Yes — leaky.** `groupby("plan").transform("mean")` is computed over the *entire* DataFrame, including the test set's rows. The feature carries information from the target back into the inputs.

The fix is to compute it on training data only, then apply it as a lookup:

```python
train_means = train_df.groupby("plan")["churned"].mean()
train_df["avg_churn_for_plan"] = train_df["plan"].map(train_means)
valid_df["avg_churn_for_plan"] = valid_df["plan"].map(train_means)
```

Or — much safer — use `category_encoders.TargetEncoder(cv=5)` inside a Pipeline, which does the right thing on each CV fold.
</details>

### Exercise 3 — ⭐⭐ A custom feature transformer

Write a tiny `LogTransformer` class that follows the sklearn estimator API (`fit`, `transform`) and applies `log1p` to the columns it's given. Plug it into the pipeline in place of the manual `log_mrr` computation.

In [ ]:
# Your code here  👇
from sklearn.base import BaseEstimator, TransformerMixin


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to every column. Stateless; fit is a no-op."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.log1p(np.asarray(X))


prep_log = ColumnTransformer([
    ("log", Pipeline([("log", LogTransformer()), ("s", StandardScaler())]), ["mrr_eur"]),
    ("num", StandardScaler(), [c for c in NUM_FEATURES if c != "log_mrr"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])
pipe_log = Pipeline([("prep", prep_log), ("model", LogisticRegression(max_iter=1000))])
print(cross_val_score(pipe_log, dfx, dfx["churned"], cv=cv, scoring="roc_auc").mean().round(3))
```

Writing custom transformers is how you keep complex preprocessing inside the
Pipeline. Once it's there, CV stays honest no matter how clever the transformation gets.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ Interaction features

Add **interaction terms** between `mrr_eur` and `last_login_days` (their product). Test whether the new feature lifts the cross-validation score of a logistic regression.


<details>
<summary>💡 <b>Solution</b></summary>

```python
dfx["mrr_x_login"] = dfx["mrr_eur"] * dfx["last_login_days"]

NEW = NUM_FEATURES + ["mrr_x_login"]
prep2 = ColumnTransformer([
    ("num", StandardScaler(), NEW),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
])
pipe2  = Pipeline([("prep", prep2), ("model", LogisticRegression(max_iter=1000))])
score2 = cross_val_score(pipe2, dfx, dfx["churned"], cv=cv, scoring="roc_auc")

print(f"Original    : {scores.mean():.3f} ± {scores.std():.3f}")
print(f"+ interaction: {score2.mean():.3f} ± {score2.std():.3f}")
```

**Why interactions help linear models.** A logistic regression
cannot, on its own, learn "*high MRR but inactive*" — that's a
non-linear combination of two features. Multiplying them gives the
model a feature whose coefficient *is* that interaction. Random
Forests find such interactions automatically; linear models need
them spelled out.

</details>

### Stretch exercise B — ⭐⭐⭐ Embedded feature selection with `SelectFromModel`

Use `sklearn.feature_selection.SelectFromModel` on top of an L1-regularised logistic regression to *automatically* drop weakly-predictive features. Compare the final feature count and CV score against the full-feature baseline.


In [ ]:
# Your code here  👇
from sklearn.feature_selection import SelectFromModel


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model    import LogisticRegression

selector = SelectFromModel(
    LogisticRegression(penalty="l1", solver="liblinear", C=0.5),
    threshold="median"
)

# Quick demo on the numerics only — for full features wrap inside the same Pipeline
X_num = StandardScaler().fit_transform(dfx[NUM_FEATURES])
selector.fit(X_num, dfx["churned"])
kept = np.array(NUM_FEATURES)[selector.get_support()]
dropped = set(NUM_FEATURES) - set(kept)

print(f"Kept    ({len(kept)}): {kept.tolist()}")
print(f"Dropped : {sorted(dropped)}")
```

**L1 shrinks weak coefficients to zero**, which makes feature
selection a side-effect of fitting. Use it when you have many
features and want a compact, interpretable model. For very
high-dimensional data, the same trick scales — it's how every
"sparse model" works under the hood.

</details>

### Stretch exercise C — ⭐⭐⭐ Datetime features for a forecast

Given a series of timestamps, extract a feature matrix with: `dayofweek`, `month`, `is_weekend`, `day_of_year`, and a cyclical `month_sin` / `month_cos` pair so the model sees December and January as neighbours.

```python
import pandas as pd
ts = pd.date_range("2024-01-01", periods=10, freq="D")
```

In [ ]:
# Your code here  👇
import pandas as pd
import numpy as np
ts = pd.date_range("2024-01-01", periods=10, freq="D")

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import pandas as pd
import numpy as np
ts = pd.date_range("2024-01-01", periods=10, freq="D")

df = pd.DataFrame({"ts": ts})
df["dayofweek"]  = df["ts"].dt.dayofweek            # 0=Mon
df["month"]      = df["ts"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
df["day_of_year"]= df["ts"].dt.dayofyear
df["month_sin"]  = np.sin(2*np.pi * df["month"] / 12)
df["month_cos"]  = np.cos(2*np.pi * df["month"] / 12)
print(df)
```

**Reasoning.** Two patterns worth knowing. (1) `dt` accessors are a one-stop shop for calendar features — you almost never need to convert through Python `datetime` objects explicitly. (2) **Cyclical encoding** with sine and cosine matters whenever a feature wraps around — month, day-of-week, hour-of-day, angle. A tree can learn that 12 and 1 are similar by splitting twice, but a linear model treats them as far apart unless you give it `sin/cos`. The same trick rescues time-of-day features in fraud-detection models — without it, midnight and 23:00 are 23 hours apart in the eyes of a linear model.
</details>

### Stretch exercise D — ⭐⭐⭐ One-hot vs target encoding

You have a high-cardinality categorical column (`product_id`) with ~100 distinct values. Compare two encodings on a regression task:

- **One-hot** (100 columns, sparse).
- **Target encoding** (1 column = mean of `y` for that category, computed inside each fold to   avoid leakage).

Report 5-fold CV RMSE for both on the synthetic dataset in the skeleton.

In [ ]:
# Your code here  👇
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
n = 2_000
product = rng.integers(0, 100, size=n)
true_mean_by_product = rng.normal(0, 5, size=100)
y = true_mean_by_product[product] + rng.standard_normal(n)
df = pd.DataFrame({"product": product, "y": y})

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
n = 2_000
product = rng.integers(0, 100, size=n)
true_mean_by_product = rng.normal(0, 5, size=100)
y = true_mean_by_product[product] + rng.standard_normal(n)
df = pd.DataFrame({"product": product, "y": y})

def rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))

kf = KFold(n_splits=5, shuffle=True, random_state=0)

ohe_errs, te_errs = [], []
for tr, te in kf.split(df):
    df_tr, df_te = df.iloc[tr], df.iloc[te]
    # one-hot
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(df_tr[["product"]])
    Xt_tr, Xt_te = enc.transform(df_tr[["product"]]), enc.transform(df_te[["product"]])
    m1 = Ridge(alpha=1.0).fit(Xt_tr, df_tr["y"])
    ohe_errs.append(rmse(df_te["y"], m1.predict(Xt_te)))
    # target encoding — fit on training fold only!
    means = df_tr.groupby("product")["y"].mean()
    fallback = df_tr["y"].mean()
    Xt_tr2 = df_tr["product"].map(means).fillna(fallback).to_numpy().reshape(-1, 1)
    Xt_te2 = df_te["product"].map(means).fillna(fallback).to_numpy().reshape(-1, 1)
    m2 = Ridge(alpha=1.0).fit(Xt_tr2, df_tr["y"])
    te_errs.append(rmse(df_te["y"], m2.predict(Xt_te2)))

print(f"one-hot   RMSE = {np.mean(ohe_errs):.3f}")
print(f"target-enc RMSE = {np.mean(te_errs):.3f}")
```

**Reasoning.** Two design points. (1) **Target encoding is computed inside the loop**, never globally — if you compute it once on the whole dataset, the test fold's target values leak into the training features and your CV score becomes optimistic. (2) The `fallback` value handles categories that appear in the test fold but not in the training fold; without it, mapping new categories returns NaN. Target encoding usually beats one-hot when cardinality is high *and* the categorical has strong predictive power — exactly this synthetic setup. On more realistic data with low-frequency categories, look up *smoothed* target encoding (`category_encoders.TargetEncoder`).
</details>

## 🎁 Bonus mini-project — A feature-engineering bake-off

Build a function `bake_off(feature_sets, X, y)` that takes a dict like `{"baseline": [...], "+ time": [...]}` and returns a DataFrame of mean ± std ROC AUC for each feature set. Use it to quantify the marginal value of each feature group.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def bake_off(feature_sets: dict, dfx, target, cv=cv):
    rows = []
    for name, num_cols in feature_sets.items():
        prep = ColumnTransformer([
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ])
        pipe = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=1000))])
        s = cross_val_score(pipe, dfx, dfx[target], cv=cv, scoring="roc_auc")
        rows.append({"feature_set": name,
                     "mean": s.mean(), "std": s.std(),
                     "n_features": len(num_cols) + 2})
    return pd.DataFrame(rows).set_index("feature_set").round(3)


feature_sets = {
    "baseline":       ["mrr_eur", "months_active", "support_tickets", "last_login_days"],
    "+ log_mrr":      ["mrr_eur", "log_mrr", "months_active", "support_tickets", "last_login_days"],
    "+ tenure":       ["mrr_eur", "log_mrr", "months_active", "support_tickets", "last_login_days", "tenure_days"],
    "+ time cyclic":  NUM_FEATURES,
}
print(bake_off(feature_sets, dfx, "churned"))
```

You get a one-line story per feature group: "adding cyclic time features lifts mean AUC by X with a Y-standard-deviation gain". Now you can argue for or against each one.
</details>

## 🧠 Key takeaways

1. **Better features beat fancier models** — most of the time, more cheaply.
2. **Categorical encoding** has three options; target encoding leaks unless you're careful.
3. **Scaling** matters for distance- and gradient-based models, not for trees.
4. **Datetime columns** are gold: extract dow, month, tenure, cyclic features.
5. **Heavy-tailed features** almost always benefit from `log1p`.
6. **Target leakage** is the single most common cause of "too-good-to-be-true" CV scores. Always question a feature that jumps your score by 10+ points.
7. **Wrap all preprocessing in a `Pipeline`** so CV is leak-proof by construction.
8. Run a **feature bake-off** to quantify each feature group's marginal value.

## ✅ Self-assessment

- [ ] Choose one-hot vs ordinal encoding for a column
- [ ] Apply a `StandardScaler` and explain when it does (and doesn't) help
- [ ] Extract 5+ features from a datetime column, including cyclic encoding
- [ ] Spot a leaky feature and propose a leak-free alternative
- [ ] Build a `ColumnTransformer` that combines numeric and categorical preprocessing
- [ ] Write a custom sklearn-compatible transformer
- [ ] Quantify the marginal value of a feature group with cross-validation

## 🚀 Next step

You've now completed Module 4 — and you have a choice:

- **Apply it first** → **Module 5 — Industry Applications** (`../05_industry_applications/17_churn_clv_retention.ipynb`): churn value, fraud, segmentation, forecasting — the classic business use cases, built from exactly the skills you just finished. (This is the spiral-path order.)
- **Or push on** → **Module 6 — AI Engineering** (`../06_ai_engineering/22_ai_workflows.ipynb`), where the metrics and discipline you just internalised get applied to a completely different kind of model. Module 5 will still be there.